# Parte 3 — Calibration (Monte Carlo + GLUE)

**Roadmap:** `notebooks/03_calibration.ipynb`  
**Estado:** plantilla lista para cuando retomen la Parte 3. Ejecutar **después** de `02_sensitivity.ipynb`.

| Pieza | Rol |
|-------|-----|
| `data/sinteticos.py` | Mismos CSV que Parte 2 |
| `src/model.py` | `saint_venant_1d` con Q aguas arriba del CSV |
| Este notebook | Monte Carlo + filtro GLUE (bandas 5–95 % en x = L) |

**Parámetros:** `n`, `S0`, `B_w` (mismos rangos que Parte 2)

**Monte Carlo:** sorteo uniforme de θ → distribución de KGE (exploración).

**GLUE:** acepta simulaciones con `KGE >= GLUE_KGE_MIN` (periodo calibración) → envolvente 5–95 % del hidrograma.

**Salidas previstas:**
- `data/synthetic/monte_carlo_resumen.csv`
- `data/synthetic/glue_envelope_xL.csv`, `glue_parametros_aceptados.csv`
- `figures/monte_carlo_kge_hist.png`, `figures/glue_credibilidad_xL.png`


In [ ]:
%load_ext autoreload
%autoreload 2

import importlib
import importlib.util
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import src.model as _model_mod

importlib.reload(_model_mod)
from src.model import saint_venant_1d

_spec = importlib.util.spec_from_file_location("sinteticos", ROOT / "data" / "sinteticos.py")
sinteticos = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(sinteticos)

PARAM_NAMES = ["n", "S0", "B_w"]
PARAMS_TRUE = [sinteticos.N_MANN, sinteticos.S0, sinteticos.B_W]
L_CANAL = sinteticos.L
NX = 100
WARMUP_SECONDS = 3600.0
BOUNDS_LO = np.array([0.01, 0.0001, 20.0])
BOUNDS_HI = np.array([0.06, 0.005, 80.0])
CAL_FRAC, VAL_FRAC = 0.60, 0.30

FIG = ROOT / "figures"
DATA = ROOT / "data" / "synthetic"
for d in (FIG, DATA):
    d.mkdir(parents=True, exist_ok=True)

FAST = True
MC_N = 200 if FAST else 50_000
GLUE_KGE_MIN = 0.5
N_JOBS = 1
USE_NOISY_CSV = False
RNG_SEED = 42

print("ROOT:", ROOT, "| FAST:", FAST, "| MC_N:", MC_N)


In [ ]:
if USE_NOISY_CSV:
    csv_name = "series_corta_ruido.csv"
else:
    csv_name = "series_corta_balance.csv"

csv_path = DATA / csv_name
if not csv_path.exists():
    print("Generando CSV con data/sinteticos.py ...")
    sinteticos.generate(str(DATA))

df = pd.read_csv(csv_path, parse_dates=["datetime"])
t_sec = (df["datetime"] - df["datetime"].iloc[0]).dt.total_seconds().to_numpy(dtype=float)
q_up = df["Q_upstream_m3s"].to_numpy(dtype=float)
q_lat = df["q_lat_m3s"].to_numpy(dtype=float) if "q_lat_m3s" in df.columns else None
q_obs = df["Q_downstream_m3s"].to_numpy(dtype=float)
nt = len(df)

mask_warm = t_sec < WARMUP_SECONDS
idx_post = np.where(~mask_warm)[0]
n_post = len(idx_post)
n_cal = max(1, int(CAL_FRAC * n_post))
n_val = max(1, int(VAL_FRAC * n_post))
n_cal = min(n_cal, n_post - n_val)
i_cal = idx_post[:n_cal]
i_val = idx_post[n_cal : n_cal + n_val]

mask_cal = np.zeros(nt, dtype=bool)
mask_val = np.zeros(nt, dtype=bool)
mask_cal[i_cal] = True
mask_val[i_val] = True

print("CSV:", csv_name, " cal=", mask_cal.sum(), " val=", mask_val.sum())


In [ ]:
def simulate_outlet(params):
    p = list(map(float, params))
    q = np.asarray(
        saint_venant_1d(
            p,
            q_upstream=q_up,
            time_seconds=t_sec,
            L=L_CANAL,
            nx=NX,
            q_lat=q_lat,
        ),
        dtype=float,
    )
    return np.maximum(q, 0.0)


def kge(obs, sim, m):
    o, s = obs[m], sim[m]
    if np.std(o) < 1e-12 or np.std(s) < 1e-12:
        return -np.inf
    r = float(np.corrcoef(o, s)[0, 1])
    return float(
        1.0
        - np.sqrt(
            (r - 1) ** 2
            + (np.std(s) / np.std(o) - 1) ** 2
            + (np.mean(s) / np.mean(o) - 1) ** 2
        )
    )


def eval_kge_cal(params):
    return kge(q_obs, simulate_outlet(params), mask_cal)


## Monte Carlo exploratorio

No filtra por desempeño: solo muestra la dispersión de KGE al sortear θ uniforme en los rangos.


In [ ]:
rng = np.random.default_rng(RNG_SEED)
samples_mc = rng.uniform(BOUNDS_LO, BOUNDS_HI, size=(MC_N, 3))

print("Monte Carlo:", MC_N, "simulaciones...")
kge_mc = np.array(
    Parallel(n_jobs=N_JOBS)(delayed(eval_kge_cal)(samples_mc[i]) for i in range(MC_N))
)

pd.DataFrame(
    {"n": samples_mc[:, 0], "S0": samples_mc[:, 1], "B_w": samples_mc[:, 2], "KGE_cal": kge_mc}
).to_csv(DATA / "monte_carlo_resumen.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
finite = kge_mc[np.isfinite(kge_mc)]
ax.hist(finite, bins=min(30, max(5, MC_N // 10)), edgecolor="k", alpha=0.75)
ax.axvline(GLUE_KGE_MIN, color="r", ls="--", label=f"umbral GLUE KGE>={GLUE_KGE_MIN}")
ax.set_xlabel("KGE (calibracion)")
ax.set_ylabel("frecuencia")
ax.set_title(f"Monte Carlo ({MC_N} sorteos uniformes)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "monte_carlo_kge_hist.png", dpi=150)
plt.show()
print("Guardado:", DATA / "monte_carlo_resumen.csv")


## GLUE — bandas de credibilidad 5–95 %

Conjuntos **conductuales**: simulaciones con `KGE_cal >= GLUE_KGE_MIN`.
La envolvente 5–95 % se calcula en cada instante (figura post warm-up).


In [ ]:
behavioral = samples_mc[kge_mc >= GLUE_KGE_MIN]
n_beh = len(behavioral)
print(f"GLUE: {n_beh} / {MC_N} conjuntos conductuales (KGE >= {GLUE_KGE_MIN})")

if n_beh == 0:
    raise RuntimeError(
        "Ningun set conductual. Baje GLUE_KGE_MIN o aumente MC_N (FAST=False)."
    )

print("Simulando hidrogramas conductuales...")
Qs = np.array(
    Parallel(n_jobs=N_JOBS)(delayed(simulate_outlet)(behavioral[i]) for i in range(n_beh))
)

p05 = np.percentile(Qs, 5, axis=0)
p50 = np.percentile(Qs, 50, axis=0)
p95 = np.percentile(Qs, 95, axis=0)

pd.DataFrame(
    {"n": behavioral[:, 0], "S0": behavioral[:, 1], "B_w": behavioral[:, 2]}
).to_csv(DATA / "glue_parametros_aceptados.csv", index=False)

pd.DataFrame(
    {
        "datetime": df["datetime"],
        "Q_obs_m3s": q_obs,
        "Q_p05_m3s": p05,
        "Q_p50_m3s": p50,
        "Q_p95_m3s": p95,
    }
).to_csv(DATA / "glue_envelope_xL.csv", index=False)

t_h = t_sec / 3600.0
m_plot = ~mask_warm
fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(t_h[m_plot], p05[m_plot], p95[m_plot], alpha=0.35, label="GLUE 5-95 %")
ax.plot(t_h[m_plot], q_obs[m_plot], "k.", ms=2, label="Q_obs")
ax.plot(t_h[m_plot], p50[m_plot], "b-", lw=1, label="mediana conductual")
ax.set_xlabel("tiempo (h)")
ax.set_ylabel("Q en x=L (m3/s)")
ax.set_title(f"GLUE — {n_beh} sets conductuales (KGE cal >= {GLUE_KGE_MIN})")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "glue_credibilidad_xL.png", dpi=150)
plt.show()

inside = (q_obs[m_plot] >= p05[m_plot]) & (q_obs[m_plot] <= p95[m_plot])
print("Observaciones dentro de la banda (post warm-up):", f"{inside.mean():.1%}")
print("Guardado:", DATA / "glue_envelope_xL.csv")
